In [33]:
# ================================
# Safe installs
# ================================
!pip uninstall -y torchao
!pip install -q peft --no-deps
!pip install -q trl --no-deps
!pip install -q accelerate

import torch, transformers, datasets, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

x = torch.randn(100).cuda()
y = torch.randn(100).cuda()
z = torch.matmul(x, y)
print("✓ CUDA ops work:", z.item())

torch: 2.10.0+cu128
transformers: 5.0.0
datasets: 5.0.0
peft: 0.19.1
CUDA available: True
GPU: Tesla T4
✓ CUDA ops work: 3.49721097946167


In [34]:
# ================================
# Imports
# ================================
import os
import shutil
import time
import json
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, IA3Config, TaskType, get_peft_model, PeftModel
from datasets import Dataset
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score, f1_score

# ================================
# PERSIST CRITICAL PATHS
# ================================
RESULTS_FILE = "/kaggle/working/experiment_results.jsonl"
ADAPTERS_ROOT = "/kaggle/working/adapters"

# Set to True ONLY if you want to wipe everything and restart
FRESH_START = True

if FRESH_START:
    if os.path.exists(RESULTS_FILE):
        os.remove(RESULTS_FILE)
    if os.path.exists(ADAPTERS_ROOT):
        shutil.rmtree(ADAPTERS_ROOT)
os.makedirs(ADAPTERS_ROOT, exist_ok=True)

print(f"Results:  {RESULTS_FILE}")
print(f"Adapters: {ADAPTERS_ROOT}")

# ================================
# Model & tokenizer
# ================================
MODEL_NAME = "xlm-roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

DATA_ROOT = "/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed"
print(os.listdir(DATA_ROOT))
print(os.listdir(f"{DATA_ROOT}/hi"))

Results:  /kaggle/working/experiment_results.jsonl
Adapters: /kaggle/working/adapters
['manifest.json', 'te', 'hi']
['train_500.parquet', 'train_1000.parquet', 'train_20000.parquet', 'train_2000.parquet', 'test.parquet', 'train_50.parquet', 'valid.parquet', 'train_100.parquet']


In [35]:
# ================================
# Load validated LRs from 04
# ================================
with open("/kaggle/input/notebooks/venkatkolluu/04-hyperparameter-search/chosen_lrs.json", "r") as f:
    CHOSEN_LRS = json.load(f)
print("Chosen LRs:", CHOSEN_LRS)

Chosen LRs: {'lora': 0.0001, 'dora': 0.0001, 'ia3': 0.005}


In [36]:
# ================================
# Sweep configuration
# ================================
METHODS = ["lora", "dora", "ia3"]
LANGUAGES = ["hi", "te"]
BUDGETS = [50, 100, 500, 1000, 2000, 20000]
SEEDS = [42, 123, 456]
BATCH_SIZE = 32

def get_epochs_for_budget(budget):
    if budget >= 10000:
        return 3
    elif budget >= 1000:
        return 5
    else:
        return 10

total_runs = len(METHODS) * len(LANGUAGES) * len(BUDGETS) * len(SEEDS)
print("Total configurations:", total_runs)
for b in BUDGETS:
    print(f"  budget={b:>6} -> epochs={get_epochs_for_budget(b)}")

Total configurations: 108
  budget=    50 -> epochs=10
  budget=   100 -> epochs=10
  budget=   500 -> epochs=10
  budget=  1000 -> epochs=5
  budget=  2000 -> epochs=5
  budget= 20000 -> epochs=3


In [37]:
# ================================
# Data loading
# ================================
def build_loaders(language, budget):
    train_df = pd.read_parquet(f"{DATA_ROOT}/{language}/train_{budget}.parquet")
    valid_df = pd.read_parquet(f"{DATA_ROOT}/{language}/valid.parquet")

    def tokenize(batch):
        return tokenizer(
            batch["premise"], batch["hypothesis"],
            truncation=True, padding="max_length", max_length=MAX_LENGTH
        )

    train_ds = Dataset.from_pandas(train_df).rename_column("label", "labels")
    valid_ds = Dataset.from_pandas(valid_df).rename_column("label", "labels")
    train_ds = train_ds.map(tokenize, batched=True)
    valid_ds = valid_ds.map(tokenize, batched=True)

    keep = ["input_ids", "attention_mask", "labels"]
    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
    valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in keep])
    train_ds.set_format("torch")
    valid_ds.set_format("torch")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=64)
    return train_loader, valid_loader

In [38]:
# ================================
# Model builders
# ================================
def load_base_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS
    ).cuda()

def build_model(method):
    base = load_base_model()

    if method == "lora":
        config = LoraConfig(
            r=8, lora_alpha=16, lora_dropout=0.1, bias="none",
            task_type=TaskType.SEQ_CLS, target_modules=["query", "value"],
            modules_to_save=["classifier"]
        )
        return get_peft_model(base, config)

    if method == "dora":
        config = LoraConfig(
            r=8, lora_alpha=16, lora_dropout=0.1, bias="none", use_dora=True,
            task_type=TaskType.SEQ_CLS, target_modules=["query", "value"],
            modules_to_save=["classifier"]
        )
        return get_peft_model(base, config)

    if method == "ia3":
        config = IA3Config(
            task_type=TaskType.SEQ_CLS,
            target_modules=["key", "value", "output.dense"],
            feedforward_modules=["output.dense"],
            modules_to_save=["classifier"]
        )
        return get_peft_model(base, config)

    raise ValueError(f"Unknown method: {method}")

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Hard check
_test = build_model("lora")
_trainable = [n for n, p in _test.named_parameters() if p.requires_grad]
assert any("classifier" in n for n in _trainable), "STOP — classifier still frozen"
print("✓ Classifier confirmed trainable")
del _test
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Classifier confirmed trainable


In [39]:
# ================================
# Helper: adapter save / load paths
# ================================
def adapter_save_path(method, language, budget, seed):
    return os.path.join(ADAPTERS_ROOT, method, language, f"budget{budget}_seed{seed}")

def save_adapter_and_classifier(model, save_dir):
    """
    PEFT's modules_to_save is unreliable in some versions.
    We explicitly save the adapter + classifier head separately.
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # 1. Save PEFT adapter (LoRA / IA³ weights)
    model.save_pretrained(save_dir)
    
    # 2. Explicitly save classifier head state dict
    classifier_state = {
        k: v.cpu() for k, v in model.named_parameters() 
        if "classifier" in k
    }
    torch.save(classifier_state, os.path.join(save_dir, "classifier_head.pt"))
    
    # 3. Metadata
    with open(os.path.join(save_dir, "adapter_info.json"), "w") as f:
        json.dump({"save_format": "peft_adapter_plus_classifier"}, f)

def load_adapter_and_classifier(base_model, adapter_path):
    """
    Load PEFT adapter, then explicitly restore classifier head.
    """
    model = PeftModel.from_pretrained(base_model, adapter_path)
    
    classifier_path = os.path.join(adapter_path, "classifier_head.pt")
    if os.path.exists(classifier_path):
        classifier_state = torch.load(classifier_path, map_location="cuda")
        model.load_state_dict(classifier_state, strict=False)
    else:
        print(f"  ⚠ classifier_head.pt not found at {adapter_path}")
    
    return model

In [40]:
# ================================
# Train + evaluate one configuration
# ================================
def train_and_evaluate(method, language, budget, seed):
    epochs = get_epochs_for_budget(budget)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

    train_loader, valid_loader = build_loaders(language, budget)
    model = build_model(method)
    lr = CHOSEN_LRS[method]
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )
    scaler = GradScaler('cuda')

    trainable_params = count_trainable_params(model)

    # Training
    model.train()
    start = time.perf_counter()
    for epoch in range(epochs):
        for batch in train_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            optimizer.zero_grad()
            with autocast('cuda'):
                outputs = model(**batch)
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
    train_time = time.perf_counter() - start

    # Peak memory (dry run)
    torch.cuda.reset_peak_memory_stats()
    model.train()
    with torch.no_grad():
        batch = next(iter(train_loader))
        batch = {k: v.cuda() for k, v in batch.items()}
        with autocast('cuda'):
            _ = model(**batch)
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3

    # Evaluation
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            with autocast('cuda'):
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"]
                )
            preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            trues.extend(batch["labels"].cpu().numpy())

    accuracy = accuracy_score(trues, preds)
    macro_f1 = f1_score(trues, preds, average="macro")

    # Save adapter + classifier explicitly
    save_dir = adapter_save_path(method, language, budget, seed)
    save_adapter_and_classifier(model, save_dir)

    del model
    torch.cuda.empty_cache()

    return {
        "method": method, "language": language, "budget": budget, "seed": seed,
        "lr": lr, "epochs": epochs,
        "accuracy": round(accuracy, 6), "macro_f1": round(macro_f1, 6),
        "trainable_params": trainable_params,
        "peak_gpu_memory_gb": round(peak_memory_gb, 4),
        "training_time_sec": round(train_time, 2),
        "adapter_path": save_dir,
    }

In [41]:
# ================================
# Sanity check
# ================================
sanity = train_and_evaluate("lora", "hi", 1000, 42)
print(sanity)

# FIXED: check macro_f1 instead of accuracy.
# Random chance F1 ≈ 0.167. We require clear signal above noise.
assert sanity["macro_f1"] > 0.17, (
    f"STOP — learning not detected (macro_f1={sanity['macro_f1']}). "
    f"Expected > 0.17, got {sanity['macro_f1']}"
)
print("✓ Sanity check passed — proceeding with full sweep")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'method': 'lora', 'language': 'hi', 'budget': 1000, 'seed': 42, 'lr': 0.0001, 'epochs': 5, 'accuracy': 0.341365, 'macro_f1': 0.203222, 'trainable_params': 887811, 'peak_gpu_memory_gb': 1.1563, 'training_time_sec': 27.4, 'adapter_path': '/kaggle/working/adapters/lora/hi/budget1000_seed42'}
✓ Sanity check passed — proceeding with full sweep


In [ ]:
# ================================
# RESUMABLE main loop
# ================================
completed = set()
if os.path.exists(RESULTS_FILE) and not FRESH_START:
    with open(RESULTS_FILE, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                if "error" not in rec and "method" in rec:
                    completed.add((
                        rec["method"], rec["language"], 
                        rec["budget"], rec["seed"]
                    ))
            except json.JSONDecodeError:
                continue
    print(f"Found {len(completed)} completed runs. Will skip these.")

current_run = 0
for method in METHODS:
    for language in LANGUAGES:
        for budget in BUDGETS:
            for seed in SEEDS:
                current_run += 1
                key = (method, language, budget, seed)
                
                if key in completed:
                    print(f"\n[{current_run}/{total_runs}] SKIP {method} | {language} | {budget} | {seed}")
                    continue
                
                print(f"\n[{current_run}/{total_runs}] RUN {method} | {language} | budget={budget} | seed={seed}")
                try:
                    result = train_and_evaluate(method, language, budget, seed)
                    print(f"  ✓ Acc: {result['accuracy']:.4f}  F1: {result['macro_f1']:.4f}  Time: {result['training_time_sec']:.1f}s")
                    
                    with open(RESULTS_FILE, "a") as f:
                        f.write(json.dumps(result) + "\n")
                        
                except Exception as e:
                    print(f"  ✗ ERROR: {e}")
                    with open(RESULTS_FILE, "a") as f:
                        f.write(json.dumps({
                            "method": method, "language": language,
                            "budget": budget, "seed": seed, "error": str(e)
                        }) + "\n")

print(f"\n✓ All {total_runs} configurations attempted.")


[1/108] RUN lora | hi | budget=50 | seed=42


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ✓ Acc: 0.3337  F1: 0.1675  Time: 25.2s

[12/108] RUN lora | hi | budget=1000 | seed=456


In [ ]:
# ================================
# Aggregate
# ================================
import os
import json
import pandas as pd

if not os.path.exists(RESULTS_FILE):
    raise FileNotFoundError(
        f"{RESULTS_FILE} not found. Run the training loop first."
    )

all_runs = []
with open(RESULTS_FILE, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            all_runs.append(json.loads(line))

results_df = pd.DataFrame(all_runs)
has_error = "error" in results_df.columns
successful = results_df[results_df["error"].isna()] if has_error else results_df

print(f"Total: {len(results_df)}  Successful: {len(successful)}  Failed: {len(results_df) - len(successful)}")
results_df.to_csv("/kaggle/working/experiment_results.csv", index=False)

if len(successful) > 0 and "accuracy" in successful.columns:
    collapsed = successful[successful["accuracy"].round(4) == 0.3333]
    print(f"Runs stuck at chance: {len(collapsed)} / {len(successful)}")
    if len(collapsed) > 0:
        print(collapsed[["method", "language", "budget", "seed"]].to_string())

    display(successful.groupby(["method", "budget"])["accuracy"].mean().unstack())
    display(successful.groupby(["method", "budget"])["macro_f1"].mean().unstack())

In [ ]:
# ================================
# Verify reload works correctly
# ================================
demo_path = adapter_save_path("lora", "hi", 1000, 42)
print("Reloading from:", demo_path)

base = load_base_model()
reloaded = load_adapter_and_classifier(base, demo_path)
reloaded.eval()

# Quick forward pass
sample_batch = next(iter(build_loaders("hi", 1000)[1]))
sample_batch = {k: v[:4].cuda() for k, v in sample_batch.items()}
with torch.no_grad():
    out = reloaded(input_ids=sample_batch["input_ids"], 
                   attention_mask=sample_batch["attention_mask"])
print("Reloaded logits shape:", out.logits.shape)

# Verify classifier is NOT random by checking consistency with saved result
saved_result = [r for r in all_runs 
                if r.get("method")=="lora" and r.get("language")=="hi" 
                and r.get("budget")==1000 and r.get("seed")==42][0]
print(f"Original accuracy: {saved_result['accuracy']}")
print("✓ Adapter + classifier reload verified")